In [1]:
import cv2
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path

import src.ModelTrain as ModelTrain
from src.VideoIngestor import VideoIngestor
from dropbox_utils.manage_temp_video import delete_temp_video, download_video_to_temp

In [4]:
F1_FIG_NAME = 'f1_score_evolution.png'
LOSS_FIG_NAME = 'loss_evolution.png'
CONF_MATRIX_NAME = 'confusion_matrix.png'

**Phase 1**

In [6]:
# This configuration is used on the baseline model, and fixed for all the experiments in the project. 
# Doing this we ensure that the evolution of the results is only due to the changes in the data.
baseline_train = ModelTrain.TrainingConfig(    
    train_ratio=0.8,
    random_state=42,
    input_size=224, # Standard input size for EfficientNet
    val_resize_size=256,
    batch_size=32,
    num_workers=0,
    num_epochs=100,

    warmup_epochs=0, # No fine-tuning phase, the whole network is being trained from the beginning
    trainable_backbone_blocks=3, # Unused on this case (warmup_epochs=0)
    enable_backbone_finetuning=True,
    full_network_finetuning=True,

    head_lr=1e-3, # Unused on this case (no fine-tuning phase)
    fine_tune_head_lr=1e-4,
    backbone_lr=1e-5,

    weight_decay=5e-4,
    dropout=0.3,
    stochastic_depth_prob=0.1,
    label_smoothing=0.02,

    scheduler_factor=0.5, # Modifies the learning rate by multiplying it by this factor when the F1-macro metric don't improve for a certain number of epochs (scheduler_patience).
    scheduler_patience=2, 
    early_stopping_patience=10,
    min_lr=1e-6, 
    gradient_clip_norm=1.0,

    use_weighted_loss=True,
    use_weighted_sampler=False, # Using also a weighted sampler overrepresents the minority classes, leading to overfitting.
    )

In [ ]:
phase2_img_dir = Path('unified_images')
phase2_csv_dir = Path('unified_data_baseline.csv')
phase2_dir = Path('baseline_experiment')

trained_model, validation_loader = ModelTrain.train(phase2_csv_dir, phase2_img_dir, phase2_dir, baseline_train)

results_path = ModelTrain.RESULTS_DIR / Path(phase2_dir)
image_paths = [results_path / F1_FIG_NAME, results_path / LOSS_FIG_NAME, results_path / CONF_MATRIX_NAME]

titles = ["F1 Score", "Loss", "Confusion Matrix"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, img_path, title in zip(axes, image_paths, titles):
    # Carga de la imagen
    img = cv2.imread(str(img_path))
    
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
    else:
        ax.text(0.5, 0.5, 'Error: Image not found', fontsize=12, color='red',ha='center', va='center', transform=ax.transAxes)
        
    ax.set_title(title, fontsize=12)
    ax.axis('off') 

plt.tight_layout()
plt.show()

NameError: name 'baseline_train' is not defined

**Phase 2**

In [2]:
def augment_dataset(
    yolo_weights_path: str | Path,
    metadata_csv_path: str | Path = "data/phase2/unified_data_baseline_phase2.csv",
    output_dir: str | Path = "data/phase2/frames",
    output_csv_path: str | Path = "data/phase2/data_phase2.csv",
    max_candidates_per_video: int = 5,
) -> dict[str, int | str]:
    metadata_csv_path = Path(metadata_csv_path)
    output_csv_path = Path(output_csv_path)

    metadata_df = pd.read_csv(
        metadata_csv_path,
        dtype=str,
        keep_default_na=False,
    )

    sorted_df = metadata_df.sort_values(
        ["patient_id", "video_filename", "elapsed_seconds", "filename"]
    ).reset_index(drop=True)

    ingestor = VideoIngestor(
        yolo_weights_path=yolo_weights_path,
        output_dir=output_dir,
        max_candidates_per_video=max_candidates_per_video,
    )
    
    grouped = sorted_df.groupby(["patient_id", "video_filename"], sort=False)
    total_videos = grouped.ngroups

    augmented_groups = []
    failed_groups = []

    for video_index, ((patient_id, video_filename), video_rows_df) in enumerate(grouped, start=1):
        print(f"\n[{video_index}/{total_videos}] Processing {patient_id} | {video_filename}")

        local_video_path = download_video_to_temp(patient_id=patient_id, video_name=video_filename)
        if local_video_path is None:
            print("Skipping video because download failed.")
            failed_groups.append(
                {
                    "patient_id": patient_id,
                    "video_filename": video_filename,
                    "video_rows_df": video_rows_df,
                    "local_video_path": None,
                }
            )
            continue

        try:
            augmented_video_df = ingestor.augment_video_rows(
                video_path=local_video_path,
                metadata_rows=video_rows_df,
            )
            augmented_groups.append(augmented_video_df)
            delete_temp_video(local_video_path)
        except Exception as error:
            print(f"\nDeferring retry for {patient_id} | {video_filename}: {error}")
            failed_groups.append(
                {
                    "patient_id": patient_id,
                    "video_filename": video_filename,
                    "video_rows_df": video_rows_df,
                    "local_video_path": local_video_path,
                }
            )

    recovered_groups = []
    still_failed_groups = []

    if failed_groups:
        print(f"\nRetrying {len(failed_groups)} failed videos after first pass...")

    for retry_index, failed_group in enumerate(failed_groups, start=1):
        patient_id = failed_group["patient_id"]
        video_filename = failed_group["video_filename"]
        video_rows_df = failed_group["video_rows_df"]
        local_video_path = failed_group["local_video_path"]

        print(f"\n[retry {retry_index}/{len(failed_groups)}] Processing {patient_id} | {video_filename}")

        if local_video_path is None:
            local_video_path = download_video_to_temp(patient_id=patient_id, video_name=video_filename)
            if local_video_path is None:
                print("Retry failed because the video could not be downloaded.")
                still_failed_groups.append(
                    {
                        "patient_id": patient_id,
                        "video_filename": video_filename,
                        "local_video_path": None,
                    }
                )
                continue

        try:
            augmented_video_df = ingestor.augment_video_rows(
                video_path=local_video_path,
                metadata_rows=video_rows_df,
            )
            recovered_groups.append(augmented_video_df)
            delete_temp_video(local_video_path)
        except Exception as error:
            print(f"\nRetry failed for {patient_id} | {video_filename}: {error}")
            print(f"Keeping local video for inspection: {local_video_path}")
            still_failed_groups.append(
                {
                    "patient_id": patient_id,
                    "video_filename": video_filename,
                    "local_video_path": local_video_path,
                }
            )

    all_groups = [*augmented_groups, *recovered_groups]
    if all_groups:
        augmented_df = pd.concat(all_groups, ignore_index=True)
    else:
        augmented_df = sorted_df.drop(columns=["elapsed_seconds", "video_filename"], errors="ignore")

    output_csv_path.parent.mkdir(parents=True, exist_ok=True)
    augmented_df.to_csv(output_csv_path, index=False, encoding="utf-8")

    return {
        "videos_processed_first_pass": len(augmented_groups),
        "videos_recovered_second_pass": len(recovered_groups),
        "videos_still_failed": len(still_failed_groups),
        "rows_output": len(augmented_df),
        "output_csv_path": str(output_csv_path),
    }

augment_dataset(r"utils/models/Kvasir_yolov8m.pt")


[1/570] Processing 165507 | 20241210_144326_R1_fe89bd6ae0bfcd70.mp4
Reusing local video: c:\Users\luis\Documents\temp_workspace\20241210_144326_R1_fe89bd6ae0bfcd70.mp4
165507 data increased 8/8 | generated 35
🗑️  File deleted: 20241210_144326_R1_fe89bd6ae0bfcd70.mp4

[2/570] Processing 289827 | 20241204_134033_R1_8f99b423ac93fa5b.mp4
⬇️  Starting download: 20241204_134033_R1_8f99b423ac93fa5b.mp4 ...
✅ Download complete: c:\Users\luis\Documents\temp_workspace\20241204_134033_R1_8f99b423ac93fa5b.mp4 (506.29 MB)
289827 data increased 6/6 | generated 30
🗑️  File deleted: 20241204_134033_R1_8f99b423ac93fa5b.mp4

[3/570] Processing 289827 | 20241204_134452_R2_8f99b423ac93fa5b.mp4
⬇️  Starting download: 20241204_134452_R2_8f99b423ac93fa5b.mp4 ...
✅ Download complete: c:\Users\luis\Documents\temp_workspace\20241204_134452_R2_8f99b423ac93fa5b.mp4 (281.59 MB)
289827 data increased 2/2 | generated 10
🗑️  File deleted: 20241204_134452_R2_8f99b423ac93fa5b.mp4

[4/570] Processing 289827 | 20241204_

{'videos_processed_first_pass': 544,
 'videos_recovered_second_pass': 2,
 'videos_still_failed': 24,
 'rows_output': 16663,
 'output_csv_path': 'data\\phase2\\data_phase2.csv'}

In [ ]:
phase2_img_dir = Path('phase2/frames')
phase2_csv_dir = Path('phase2/data_phase2.csv')
phase2_dir = Path('phase2')

trained_model, validation_loader = ModelTrain.train(phase2_csv_dir, phase2_img_dir, phase2_dir, baseline_train)

results_path = ModelTrain.RESULTS_DIR / Path(phase2_dir)
image_paths = [results_path / F1_FIG_NAME, results_path / LOSS_FIG_NAME, results_path / CONF_MATRIX_NAME]

titles = ["F1 Score", "Loss", "Confusion Matrix"]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, img_path, title in zip(axes, image_paths, titles):
    # Carga de la imagen
    img = cv2.imread(str(img_path))
    
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
    else:
        ax.text(0.5, 0.5, 'Error: Image not found', fontsize=12, color='red',ha='center', va='center', transform=ax.transAxes)
        
    ax.set_title(title, fontsize=12)
    ax.axis('off') 

plt.tight_layout()
plt.show()

Hardware assigned for tensor computations: cuda
Train class distribution:
histology
Adenoma                     7351
Sessile_serrated_adenoma    3278
Hyperplastic                1788
Adenocarcinoma               467

Validation class distribution:
histology
Adenoma                     1952
Sessile_serrated_adenoma     928
Hyperplastic                 311
Adenocarcinoma               141

Loss weights: {'Adenoma': 0.471, 'Sessile_serrated_adenoma': 0.7053, 'Hyperplastic': 0.955, 'Adenocarcinoma': 1.8687}
Initiating training phase. Saving results to: C:\Users\luis\Documents\TFG - Data-Centric AI\results\phase2


Training Progress:   0%|          | 0/100 [00:00<?, ?epoch/s]


Switching to full-network fine-tuning at epoch 1.


Training Progress:   1%|          | 1/100 [09:40<15:58:06, 580.68s/epoch, Stage=full_network, Train Loss=1.2145, Val Loss=1.0946, Val F1=0.4344 [saved], LR=1.0e-05/1.0e-04]